# 03 · Tarea: la ciudad tiene ritmo semanal

**Entrega:** este notebook ejecutado (con tus gráficas) y las respuestas a las 4 preguntas del final, en las celdas indicadas.

En clase modelamos una ciudad cuya demanda sube por una ola de calor. Ahora añadimos algo real:
**los fines de semana la demanda baja** (industria y oficinas cierran). Vas a comprobar que un solo kernel RBF
ya no describe bien los datos, y que la solución es **sumar un segundo kernel**, uno periódico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
AZUL, ORO, GRIS = "#003D79", "#D59F0F", "#8A8F98"   # colores UNAM
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

In [ ]:
def demanda_ciudad(dias=35, semilla=30, ritmo_semanal=True):
    rng = np.random.default_rng(semilla)
    t = np.arange(dias, dtype=float)
    tendencia = 120 + 25 / (1 + np.exp(-(t - 14) / 4))
    vaiven    = 5 * np.sin(2 * np.pi * t / 14)
    fin_de_semana = -15 * np.isin(t % 7, [5, 6]) if ritmo_semanal else 0   # sábado y domingo bajan 15 MW
    return t, tendencia + vaiven + fin_de_semana + rng.normal(0, 2.5, dias)

t, y = demanda_ciudad()
observado = np.ones(len(t), dtype=bool); observado[28:] = False
X, Y = t[observado], y[observado]
x = np.linspace(0, 34, 200); futuro = x >= 28

plt.plot(X, Y, "o-", color=AZUL, lw=0.8); plt.axvspan(27.5, 34.5, color=ORO, alpha=0.15)
for s in range(5, 28, 7): plt.axvspan(s - 0.5, s + 1.5, color=GRIS, alpha=0.2)
plt.xlabel("día"); plt.ylabel("MW"); plt.title("Ahora con fines de semana (franjas grises)"); plt.show()

## Parte 1 · Un solo kernel (el de la clase)

Ajusta el GP con scikit-learn dejando que aprenda los hiperparámetros, como al final de la clase.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, ExpSineSquared

b = Y.mean()
k_rbf = ConstantKernel(Y.var()) * RBF(length_scale=5.0, length_scale_bounds=(1.0, 30.0))   # ℓ entre 1 y 30 días
gp1 = GaussianProcessRegressor(kernel=k_rbf, alpha=2.5**2, n_restarts_optimizer=5, random_state=0).fit(X[:, None], Y - b)
m1, s1 = gp1.predict(x[:, None], return_std=True); m1 += b
print("kernel aprendido:", gp1.kernel_)

def dibuja(m, s, titulo):
    plt.fill_between(x, m - 2*s, m + 2*s, color=AZUL, alpha=0.15, label="banda 95 %")
    plt.plot(x, m, color=AZUL, lw=2, label="media"); plt.plot(X, Y, "o", color=AZUL, label="datos")
    plt.plot(t[~observado], y[~observado], "o", color=ORO, label="lo que pasó")
    plt.xlabel("día"); plt.ylabel("MW"); plt.legend(loc="upper left", fontsize=9); plt.title(titulo); plt.show()

dibuja(m1, s1, "Un solo kernel RBF")

## Parte 2 · Dos kernels: RBF + periódico

El kernel **periódico** (`ExpSineSquared`) dice: "un día se parece a otro si están a un múltiplo de 7 días de distancia".
Los kernels **se suman**: $k = k_{\text{RBF}} + k_{\text{per}}$.

Completa la celda: crea `k_dos` sumando `k_rbf` y un `ExpSineSquared` con `periodicity=7.0` (fíjalo con `periodicity_bounds="fixed"`
para que scikit no lo cambie) multiplicado por un `ConstantKernel`.

In [ ]:
# TODO: completa el kernel compuesto
k_per = ConstantKernel(50.0) * ExpSineSquared(length_scale=1.0, periodicity=7.0, periodicity_bounds="fixed")
k_dos = ...   # ← suma de k_rbf y k_per

gp2 = GaussianProcessRegressor(kernel=k_dos, alpha=2.5**2, n_restarts_optimizer=5, random_state=0).fit(X[:, None], Y - b)
m2, s2 = gp2.predict(x[:, None], return_std=True); m2 += b
print("kernel aprendido:", gp2.kernel_)
dibuja(m2, s2, "RBF + periódico (7 días)")

## Parte 3 · Compara la capacidad a reservar

In [ ]:
for nombre, m, s in [("RBF", m1, s1), ("RBF + periódico", m2, s2)]:
    print(f"{nombre:<17} pico esperado {m[futuro].max():6.1f} MW   reserva 95 % {(m + 2*s)[futuro].max():6.1f} MW   "
          f"log-verosimilitud {(gp1 if nombre == 'RBF' else gp2).log_marginal_likelihood_value_:7.1f}")

## Parte 4 · Preguntas (responde en las celdas)

**P1.** Mirando las dos gráficas: ¿qué hace el modelo de un solo kernel con los fines de semana? ¿Y el de dos kernels?

*Tu respuesta:* 

**P2.** ¿Qué modelo pide una reserva más chica? Mira los puntos dorados: ¿le habría alcanzado? ¿Qué le pasó al modelo de un kernel con los días entre semana?

*Tu respuesta:* 

**P3.** La *log-verosimilitud marginal* mide qué tan bien explica el modelo los datos (más alta = mejor). ¿Cuál gana? ¿Coincide con lo que ves?

*Tu respuesta:* 

**P4 (reto, opcional).** En clase escribimos el GP desde cero con `k(a, c)`. Escribe `k_dos_manual(a, c)` que sume el RBF y un periódico
$\sigma_p^2 \exp\!\big(-2\sin^2(\pi (t-t')/7)/\ell_p^2\big)$ con los valores que aprendió `gp2.kernel_`, repite las seis líneas de álgebra
y comprueba que obtienes la misma media que scikit-learn.

In [ ]:
# tu código aquí (opcional)

